# 📋 Odpowiedzi: Model Języka polskie nazwy miast

*Wygenerowano automatycznie — nie commitować!*

---
### 1. Zadanie 1 — wybierz miasta

Z DataFrame `df_cities` wybierz tylko te wiersze, gdzie kolumna `Rodzaj miejscowości` zawiera wartość `'miasto'`. Z tego podzbioru wyciągnij kolumnę `Główna nazwa miejscowości` jako **listę Pythonową** i zapisz do zmiennej `list_of_cities`.

In [ ]:
list_of_cities = df_cities[df_cities['Rodzaj miejscowości'] == 'miasto']['Główna nazwa miejscowości'].tolist()

list_of_cities[:10]

---
### 2. Zadanie 2 — dodaj tokeny do każdej nazwy

Dla każdej nazwy w `list_of_cities` utwórz nowy string poprzez dodanie `%` na początku i `!` na końcu. Wynik zapisz do listy `city_list`. Polecam użyć *list comprehension*.

In [ ]:
city_list = ['%' + city + '!' for city in list_of_cities]

city_list[:10]

---
### 3. Zadanie 3 — uzupełnij klasę LanguageModel

Uzupełnij metodę `__init__` zgodnie z opisem powyżej. Trzy warstwy: `Embedding`, `GRU`, `Dense`. Pamiętaj o flagach `mask_zero`, `return_sequences`, `return_state` oraz odpowiedniej funkcji aktywacji w warstwie wyjściowej.

In [ ]:
reset_seed(SEED)


class LanguageModel(keras.Model):
    def __init__(self):
        super().__init__()

        # L config (best z hyperparam search) — emb=256, gru=128, dropout=0.1 (~172k params)
        self.embedding = Embedding(vocab_size, output_dim=256, mask_zero=True)
        self.rnn = GRU(128, return_sequences=True, return_state=True, dropout=0.1)
        self.fc = Dense(vocab_size, activation='softmax')

    def call(self, inputs, training=False, hidden_state=None, return_state=False):
        x = self.embedding(inputs, training=training)

        # initial_state=None → GRU automatycznie użyje wektora zerowego
        seq_out, last_hidden_state = self.rnn(x, training=training, initial_state=hidden_state)
        x = self.fc(seq_out)

        if return_state:
            return x, last_hidden_state
        return x


model = LanguageModel()
model(X_train)
model.summary()

---
### 4. Zadanie 4 — skompiluj i wytrenuj model

Trzy kroki:

1. **`criterion`** — utwórz instancję **Sparse Categorical Cross-Entropy**.
2. **`model.compile(...)`** — przekaż `loss=criterion` oraz `optimizer=Adam(learning_rate=0.001)`.
3. **`history = model.fit(...)`** — przekaż `X_train`, `y_train`, `epochs=80`, `batch_size=64`.

Wynik (`history`) zapisuje stratę z każdej epoki — wykorzystamy to za chwilę do narysowania krzywej uczenia.

In [ ]:
reset_seed(SEED)

criterion = SparseCategoricalCrossentropy()

# L config — Adam z learning_rate=0.001 (default Adam), batch_size=64
model.compile(loss=criterion, optimizer=keras.optimizers.Adam(learning_rate=0.001))

history = model.fit(X_train, y_train, epochs=80, batch_size=64)

---
### 5. Zadanie 5 — porównaj greedy vs stochastic sampling

Dwie strategie próbkowania mają bardzo różne właściwości — sprawdź to empirycznie.

Wygeneruj **20 nazw** używając `sample_model_optimal` w trybie `greedy=True` (bez wizualizacji, bez printowania) oraz osobno **20 nazw** w trybie `greedy=False` (stochastycznie). Wszystkie startujące z prompta `"%"`.

Następnie:
1. Policz **ile unikalnych nazw** jest w każdym zbiorze.
2. Wypisz wyniki.

Co spodziewasz się zobaczyć? Zastanów się **dlaczego** greedy zawsze daje to samo, a stochastic — różne nazwy.

In [ ]:
reset_seed(SEED)

greedy_names = [sample_model_optimal("%", greedy=True, print_outcome=False) for _ in range(20)]
stochastic_names = [sample_model_optimal("%", greedy=False, print_outcome=False) for _ in range(20)]

print(f"Greedy:     {len(set(greedy_names))} unikalnych z 20")
print(f"Stochastic: {len(set(stochastic_names))} unikalnych z 20")

print("\nPrzykładowe nazwy greedy:")
for name in greedy_names[:5]:
    print(f"  {name[1:-1]}")

print("\nPrzykładowe nazwy stochastic:")
for name in stochastic_names[:5]:
    print(f"  {name[1:-1]}")